# Probability of Optimal Action of $\epsilon$-Greedy 

The purpose of this lab is to demonstrate the effects of epsilon on the probability of choosing the most optimal action.

## Game Setup - Description

People are often given the choice of two (or more) actions (e.g. cooking or takeout, blue shirt or red shirt, etc). Rewards are always changing and often subject to feeling (e.g. the takeout was delicious but the budget is hurting). This scenario presents a very simple and objective game for a person (agent) to take action. The person gets to choose from elements in a an array and rewarded with numbers on a Mac (1 for good and 0 for bad). The most optimal choice is obviously the second element which rewards 1 point 10% of the time, while the first element only provides a reward 5% of the time. 

## Experiment

We want to see what the how probability of exploration ($\epsilon$) affects the agent's decision to choose the most optimal decision. This will require setting up an agent to choose the most optimal action given the probability of exploration and and environment to grant rewards. Then a simulation will take place to trial the agent's decision making over time (iterations). 

Data to collected regarding the frequentist's probability of the agent making the correct decision through out time. Five probabilities will be explored ranging from 0 to 1, each trialed for 1000 steps. 

In [49]:
import numpy as np

def env(a: int, p: list) -> int:
    '''Determines reward: algorithm 2-2'''

    return np.random.binomial(1, p[a])

class Agent:
    def __init__(self, epsilon: float, p: list[float]):
        self.epsilon = epsilon
        self.r = [0, 0]
        self.n_a = [0, 0]
        self.chose_most_optimal = 0
        
    def choose_action(self) -> int:
        '''Decides what action to take'''

        if np.random.binomial(1, self.epsilon) == 1:
            # Agent chooses to explore
            a = np.random.choice([0, 1])
        else:
            max_val = np.max(self.r)
            if len(max_vals := np.where(self.r == max_val)[0]) == 2:
                a = np.random.choice(max_vals)
            else:
                # Agent chooses the most optimal solution
                a = int(np.argmax(self.r))

        # The agent chose the optimal either randomly or greedily
        if a == 1:
            self.chose_most_optimal += 1

        # Increment the number of actions
        self.n_a[a] += 1

        return a
    
    def update_reward(self, r: int, a: int):
        '''Updates the average reward'''

        self.r[a] = self.r[a] + a / self.n_a[a] * (r - self.r[a])
    

In [ ]:

def trial(epsilon: float) -> list:
    '''Runs a trial at a given exploration probability'''

    # Step 1 - Initialize reward states
    p = [0.05, 0.10]

    # Step 2 - Initialize agent
    agent = Agent(epsilon, p)

    # Collect results
    data = []

    # Step 3 - Loop for ever (jk for 10,000 loops)
    for i in range(1, 1001):

        a = agent.choose_action()

        # Step 5 - Receive reward
        r = env(a, p)

        # Step 6 - Calculate the average reward
        agent.update_reward(r, a)

        # Step 7 - Update the data
        data.append({
            "Probability": agent.chose_most_optimal / i, 
            "Steps": i, 
            "epsilon": epsilon
            })

    return data 


# Set a seed for reproduceability
np.random.seed(42)

# Trial
data = []
for _ in range(100):
    for epsilon in [0.0, 0.25, 0.50, 0.75, 1.00]:
        data += trial(epsilon)


## Results

This experiment shows the results of trialing five rates of exploration (0%, 25%, 50%, 75%, and 100%) and recodring its affects on choosing the optimal action. Each exploration rate was used to train a new agent to choose one of two options over the course of 1000 steps. Agents chose options 1 and 2 which returned a reward of one 5% and 10% of the time, respectively, else 0. Exploration determined the probability the agent "explored" and randomly chose from either option or greedily chose the most optimal option based on prior rewards. In the chance the agent had to pick two actions with equally likely rewards, the actions were chosen randomly. 

Below is a chart representing the results, averaged over 100 simulations. What is of interest in this are the blue and green lines representing an exploration rate of 0% and 100%, respectively. As the exploration rate decreases, the agent more greedily chooses the most optimal action. In fact, when the rate is 100%, the rate of choosing the most optimal action converges to 1. Meanwhile, when the exploration rate is 0%, the agent's rate of choosing the most optimal solution converges to 50%. 

In [55]:
import altair as alt
import pandas as pd

# Build data frame and average probability per step number
df = pd.DataFrame(data)
df = df.groupby(['epsilon', 'Steps']).mean().reset_index()

alt.Chart(df).mark_line().encode(
    x=alt.X('Steps'),
    y=alt.Y('Probability'),
    color = alt.Color('epsilon:N')
)

alt.Chart(...)

In [68]:
e = 0.5
p1 = e * 0.5 + (1 - e) * 0.5
p2 = e * 0.5 + (1 - e) * p1 + (1 - e) * 0.5
p3 = e * 0.5 + (1 - e) * p2 + (1 - e) * 0.5
p3

k = [{'Step': 1, 'Probability': e * 0.5 + (1 - e) * 0.5}]
for i in range(2, 11):
    k.append({
        'Step': i, 
        'Probability': e * 0.5 + (1 - e) * 0.5 + (1 - e) * k[-1]['Probability']
        })

df = pd.DataFrame(k)
alt.Chart(df).mark_line().encode(
    x = alt.X('Step'),
    y = alt.Y('Probability', axis=alt.Axis(format='%'))
)

alt.Chart(...)

## Discussion

These results are promising and show what is expected and that is that the agent, given a smaller chance to explore, will more likely choose the most optimal action. This is most evident with the blue line which represents an exploration rate of 0%. The blue line begins at 50% correct and eventually converges to 100%. Meanwhile, the agent that is allowed to explore 100% of the time, has a constant probability of 50%. This agent is essentially flipping a coin every time it chooses an action. 

All agents start learning with a probability of making the correct action 50% of the time. This happens for several reasons. The first is that, unless the exploration rate is zero, there is a chance the agent will flip a coin (random choose) between all possible actions which give it a 50% chance of being correct. Secondly, even if the agent gets to greedily pick the first time, it has pick the action with the highest average reward. However, all rewards are initialized at zero, meaning the agent still has to randomly choose an action. Finally, this is more on this particular experiment, the rewards for choosing the optimal action are both very small <50%. This means that even if the agent randomly chooses the correct action the first time, they have a higher chance of the reward being zero, which results in both average rewards $r^{avg}$ being zero, forcing the agent to random choose again.

In fact, we can see in the second chart, the true probability the 